# SPML Task 1 - Base Task

training a neural net on Fashion-MNIST using PyTorch. the architecture has two branches with a skip connection and then concatenates them before the final output layer.

In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms

# setting seed so results are reproducible
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('using:', device)

## loading the dataset

using torchvision to grab Fashion-MNIST directly. splitting 10% off training as validation.

In [ ]:
BATCH_SIZE = 64
EPOCHS = 20
LR = 0.001

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# normalising to [-1, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

full_train = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_data  = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

val_size = int(0.1 * len(full_train))
train_size = len(full_train) - val_size
train_data, val_data = random_split(full_train, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'train: {len(train_data)}, val: {len(val_data)}, test: {len(test_data)}')

In [ ]:
# just quickly checking what the images look like
imgs, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(imgs[i].squeeze(), cmap='gray')
    ax.set_title(class_names[labels[i]], fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.savefig('sample_images.png', dpi=120)
plt.show()

## the model

built according to the diagram in the task sheet. basically:
- flatten the 28x28 input to 784, then a hidden layer brings it down to 16
- from there it splits into two branches (A and B)
- branch A: 16->8 then 8->8, with a skip connection (add the first layer output to the second)
- branch B: 16->12 then 12->8
- concatenate both branch outputs (8+8=16) and pass to final output layer (16->10)

In [ ]:
class FashionNet(nn.Module):
    def __init__(self):
        super(FashionNet, self).__init__()

        self.flatten = nn.Flatten()

        # shared part before the branches
        self.fc_stem = nn.Linear(784, 16)

        # branch A layers
        self.fc_a1 = nn.Linear(16, 8)
        self.fc_a2 = nn.Linear(8, 8)

        # branch B layers
        self.fc_b1 = nn.Linear(16, 12)
        self.fc_b2 = nn.Linear(12, 8)

        # final output - concat gives us 16 inputs
        self.fc_out = nn.Linear(16, 10)

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc_stem(x))  # shared hidden layer

        # branch A with skip connection
        a1 = self.relu(self.fc_a1(x))
        a2 = self.relu(self.fc_a2(a1))
        skip = a1 + a2  # element-wise add (both are size 8 so this works directly)

        # branch B
        b1 = self.relu(self.fc_b1(x))
        b2 = self.relu(self.fc_b2(b1))

        # merge and predict
        merged = torch.cat([skip, b2], dim=1)
        out = self.fc_out(merged)
        return out


model = FashionNet().to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f'total parameters: {total_params}')

## training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# reduce lr if val loss stops improving
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5, verbose=True)

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
train_losses, val_losses = [], []
train_accs, val_accs = [], []

best_val_loss = float('inf')
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion)

    scheduler.step(vl_loss)

    train_losses.append(tr_loss)
    val_losses.append(vl_loss)
    train_accs.append(tr_acc)
    val_accs.append(vl_acc)

    # save checkpoint if this is the best so far
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_epoch = epoch
        torch.save(model.state_dict(), 'best_model.pt')

    print(f'epoch {epoch:02d}/{EPOCHS} | train loss: {tr_loss:.4f} acc: {tr_acc*100:.2f}% | val loss: {vl_loss:.4f} acc: {vl_acc*100:.2f}%')

print(f'\nbest val loss was at epoch {best_epoch}')

## plotting loss and accuracy

In [ ]:
epochs_range = range(1, EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, train_losses, label='train')
ax1.plot(epochs_range, val_losses, label='val')
ax1.axvline(best_epoch, color='gray', linestyle='--', label=f'best ({best_epoch})')
ax1.set_xlabel('epoch')
ax1.set_ylabel('loss')
ax1.set_title('Loss')
ax1.legend()

ax2.plot(epochs_range, [a*100 for a in train_accs], label='train')
ax2.plot(epochs_range, [a*100 for a in val_accs], label='val')
ax2.axvline(best_epoch, color='gray', linestyle='--', label=f'best ({best_epoch})')
ax2.set_xlabel('epoch')
ax2.set_ylabel('accuracy (%)')
ax2.set_title('Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## saving model weights using pickle

In [ ]:
# load the best checkpoint
model.load_state_dict(torch.load('best_model.pt', map_location=device))
model.eval()

with open('model_weights.pkl', 'wb') as f:
    pickle.dump(model.state_dict(), f)

print('weights saved to model_weights.pkl')

# quick check that it loads back fine
with open('model_weights.pkl', 'rb') as f:
    loaded = pickle.load(f)

test_model = FashionNet().to(device)
test_model.load_state_dict(loaded)
print('loaded back successfully')

## generating submission.csv

In [ ]:
all_preds = []
all_true = []

model.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        preds = out.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(labels.numpy())

submission = pd.DataFrame({
    'Id': range(len(all_preds)),
    'Predicted': all_preds,
    'Label': [class_names[p] for p in all_preds]
})

submission.to_csv('submission.csv', index=False)
print(f'saved submission.csv with {len(submission)} rows')
submission.head()

## test accuracy

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion)
print(f'test loss: {test_loss:.4f}')
print(f'test accuracy: {test_acc*100:.2f}%')

In [ ]:
# per class breakdown
class_correct = [0] * 10
class_total = [0] * 10

model.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        preds = out.argmax(dim=1)
        for t, p in zip(labels, preds):
            class_correct[t] += (t == p).item()
            class_total[t] += 1

for i, name in enumerate(class_names):
    acc = 100 * class_correct[i] / class_total[i]
    print(f'{name:<15}: {acc:.1f}%')

In [ ]:
from sklearn.metrics import confusion_matrix
import itertools

cm = confusion_matrix(all_true, all_preds)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)

ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(class_names, fontsize=8)

thresh = cm.max() / 2
for i, j in itertools.product(range(10), range(10)):
    ax.text(j, i, cm[i, j], ha='center', va='center',
            color='white' if cm[i, j] > thresh else 'black', fontsize=7)

ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# checking all output files are there
for f in ['model_weights.pkl', 'submission.csv', 'training_curves.png', 'confusion_matrix.png', 'sample_images.png']:
    exists = os.path.exists(f)
    size = os.path.getsize(f) if exists else 0
    print(f"{'ok' if exists else 'MISSING'} - {f} ({size} bytes)")